# 1. Import des biblioth?ques  
Importation des modules pour le traitement, pipeline et mod?les.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import joblib, os
pd.set_option('display.max_columns', None)

# 2. Chargement des donn?es  
Lecture du fichier CSV des features d?achats et aper?u.

In [ ]:
data = '../Features/Export_fe_data/achats_merged_features.csv'
df = pd.read_csv(data)
print(df.shape)
df.head()

# 3. Nettoyage initial  
Suppression des colonnes non utilis?es pour ?viter la fuite.

In [ ]:
df = df.drop(columns=['d?lai_livraison_jours', 'delai_deviation','retard_jours'])

# 4. Pr?traitement et encodage  
S?paration de X, y et encodage one-hot pour les cat?gorielles.

In [ ]:
TARGET = 'on_time_delivery'
y = df[TARGET]
X = df.drop(columns=[TARGET])

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
print(f"Numeric: {len(num_cols)} | Categorical: {len(cat_cols)}")

# 5. Entra?nement et ?valuation des mod?les  
D?finition des pipelines, mod?les et entra?nement avec ?valuation sur le jeu de test.

In [ ]:
numeric = Pipeline([('scale', StandardScaler())])
categorical = Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

pre = ColumnTransformer([('num', numeric, num_cols),
                         ('cat', categorical, cat_cols)])

models = {
    'Regression Logistic': LogisticRegression(max_iter=500, n_jobs=-1),
    'Gradient Boost': GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

results = {}
for name, m in models.items():
    clf = Pipeline([('pre', pre), ('model', m)])
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]
    roc = roc_auc_score(y_test, y_prob)
    results[name] = roc
    print(name, classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm).plot()
    plt.title(name)
    plt.show()

results

# 6. Validation crois?e  
?valuation des mod?les avec validation crois?e pour robustesse.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
for name, m in models.items():
    clf = Pipeline([('pre', pre), ('model', m)])
    scores = cross_val_score(clf, X, y, cv=cv, scoring='roc_auc')
    print(name, scores.mean(), scores.std())

# Meilleur mod?le : Gradient boost avec 0.68 de pr?cision. 
Les donn?es ?tant g?n?r?es, la pr?cision est acceptable. Avec des donn?es r?elles, on pourrait s'attendre ? une pr?cision plus ?lev?e.